# 7 · Widgets

A widget is the control a person edits an input with. Every input names its own; no type has a default.

In [ ]:
import json
from typing import Annotated, Literal

from conductor import CompiledGraph, Graph, Input, NodeRegistry, Param, Result, Series, Table, node
from conductor.widgets import Choice, DatePicker, Dropdown, FileUpload, ListWidget, NumberWidget, Range, Switch, Textarea, TextWidget, Widget
from pydantic import TypeAdapter


## Widgets with constraints

The widget goes on the `Param`, beside the `title`, `description` and `connectable` that belong to the field.

In [ ]:
PRIORITIES = (
    Choice(id="low", title="Low"),
    Choice(id="normal", title="Normal"),
    Choice(id="high", title="High"),
)


@node("send-email", title="Send Email", description="Send an email", category="mail")
def send_email(
    to: Annotated[str, Param(title="Recipient", pattern=r"^[^@]+@[^@]+$", widget=TextWidget())],
    subject: Annotated[str, Param(max_length=200, widget=TextWidget())],
    body: Annotated[str, Param(min_length=10, widget=Textarea(rows=8))],
    priority: Annotated[str, Param(widget=Dropdown(choices=PRIORITIES))] = "normal",
    retries: Annotated[int, Param(title="Retry attempts", ge=0, le=5, widget=Range(step=1))] = 1,
) -> Annotated[str, Result(title="Status")]:
    return f"sent: {to=}, {subject=}, {priority=}, {retries=}"


for inp in send_email.versions[1].interface.inputs:
    print(f"  {inp.name:<10} kind={inp.widget.kind:<10} title={inp.title!r}  bounds={dict(inp.bounds)}  required={inp.required}")


## Every widget, and one of your own

`Param(connectable=False)` closes an input to edges, and it may then be any type pydantic can validate. A `Series[X]` input takes `ListWidget`, or no widget when an edge fills it.

The limits a value must keep to are on the `Param`, in pydantic's words, and they are enforced. A control conductor does not ship is a `Widget` subclass in your own code, next to the component that renders it.


In [ ]:
class EntityDropdown(Widget):
    """Choices the host resolves. Conductor carries it to the editor and never reads it."""

    kind: Literal["entity-dropdown"] = "entity-dropdown"
    entity_kind: str = ""


@node("widget-showcase", title="Showcase", description="Every widget", category="demo")
def showcase(
    # Text
    heading: Annotated[str, Param(title="Title", max_length=100, widget=TextWidget())],
    notes: Annotated[str, Param(widget=Textarea(rows=6))],
    # Choice
    style: Annotated[str, Param(widget=Dropdown(choices=(Choice(id="a", title="A"), Choice(id="b", title="B"))))],
    user: Annotated[str, Param(widget=EntityDropdown(entity_kind="user"))],
    # Numeric
    count: Annotated[int, Param(ge=0, widget=NumberWidget(integer_only=True))],
    volume: Annotated[float, Param(ge=0, le=1, widget=Range(step=0.01))],
    # Boolean
    enabled: Annotated[bool, Param(widget=Switch())],
    # Date & file
    starts: Annotated[str, Param(widget=DatePicker(seed="today"))],
    upload: Annotated[str, Param(widget=FileUpload(accept=(".pdf", ".csv"), max_size_mb=10))],
    # Collections
    items: Annotated[Series[str], Param(min_length=1, max_length=10, widget=ListWidget())],
    sources: Annotated[Series[str], Param()],
    # Structured
    table: Annotated[Table, Param(connectable=True)],
    config: Annotated[dict, Param(connectable=False, widget=Textarea())],
) -> Annotated[str, Result(title="Summary")]:
    return "all good"


inputs = showcase.versions[1].interface.inputs
print(f"{len(inputs)} inputs, {len({inp.widget.kind for inp in inputs if inp.widget})} distinct widget kinds:\n")
for inp in inputs:
    closed = "" if inp.connectable else "  (closed to edges)"
    kind = inp.widget.kind if inp.widget else "none"
    print(f"  {inp.name:<13} {inp.dtype.id:<8} widget={kind}{closed}")


## The dump

A widget dumps through pydantic, its own fields and `kind`, for an editor to read.


In [ ]:
w = Range(step=5)
print(json.dumps(w.model_dump(mode="json"), indent=2))


In [ ]:
# The same widget as its Input carries it: title and description sit beside it.
volume = next(inp for inp in inputs if inp.name == "volume")
print(json.dumps(TypeAdapter(Input).dump_python(volume, mode="json"), indent=2))

## Running a graph

A typed-in value is checked against its input's type.

In [ ]:
@node("label", title="Label", description="Glue", category="demo")
def label(
    n: Annotated[int, Param(title="N", widget=NumberWidget(integer_only=True))],
    name: Annotated[str, Param(title="Label", widget=TextWidget())],
) -> Annotated[str, Result(title="Out")]:
    return f"{name}={n}"


compiled = CompiledGraph.from_graph(Graph(nodes=[label.graph_node("n", n=42, name="answer")]), NodeRegistry([label]))
compiled.results(await compiled.run())["n"]["result"]